In [ ]:
import numpy as np


class Tensor:

    def __init__(self, data, children=()):
        self.data = data
        self.children = set(children)
        self.grad = np.zeros(data.shape)
        self._backward = lambda : None
        self.shape = data.shape

    def __add__(self, other):
        out = Tensor(self.data + other.data, (self, other))

        def _backward():
            self.grad += unbroadcast(out.grad, self.grad)
            other.grad += unbroadcast(out.grad, other.grad)
        out._backward = _backward

        return out

    def __mul__(self, other):
        out = Tensor(self.data * other.data, (self, other))

        def _backward():
            self.grad += unbroadcast(out.grad * other.data, self.grad)
            other.grad += unbroadcast(out.grad * self.data, other.grad)
        out._backward = _backward

        return out

    def __neg__(self):
        out = Tensor(-self.data, (self,))
            
        def _backward():
            self.grad -= out.grad
        out._backward = _backward

        return out

    def __sub__(self, other):
        return self + -other

    def __pow__(self, other):
        out = Tensor(self.data ** other, (self,))

        def _backward():
            self_grad = other * self.data ** (other - 1) * out.grad
            self.grad += unbroadcast(self_grad, self.grad)
        out._backward = _backward

        return out

    def __matmul__(self, other):
        out = Tensor(self.data @ other.data, (self, other))

        def _backward():
            self.grad += unbroadcast(out.grad @ other.data.T, self.grad)
            other.grad += unbroadcast(self.data.T @ out.grad, other.grad)
        out._backward = _backward

        return out

    def __truediv__(self, other):
        return self * (other ** -1)

    def backward(self, R=None):
        topo = []
        visited = set()
        
        def topo_sort(node):
            if node in visited:
                return
            visited.add(node)
            for child in node.children:
                topo_sort(child)
            topo.append(node)

        self.grad = np.ones(self.shape) if R is None else R
        topo_sort(self)
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
            return f'{self.data}'

def unbroadcast(input, target):
    target_shape = target.shape

    while len(target.shape) < len(input.shape):
        target = np.expand_dims(target, axis=0)
    for axis, dim in enumerate(target.shape):
        if dim == 1:
            input = np.sum(input, axis=axis, keepdims=True)

    return input.reshape(target_shape)

def gradcheck(f, inputs):
    inputs = [Tensor(input.data.astype(np.float64)) for input in inputs]
    out = f(*inputs)
    R = np.random.rand(*out.shape)
    out.backward(R)

    EPS = 1e-6
    for i in range(len(inputs)):
        inputs_pos = inputs.copy()
        inputs_neg = inputs.copy()
        for idx in np.ndindex(inputs[i].shape):
            input_pos = inputs[i].data.copy()
            input_neg = inputs[i].data.copy()
            input_pos[idx] += EPS
            input_neg[idx] -= EPS
            inputs_pos[i] = Tensor(input_pos)
            inputs_neg[i] = Tensor(input_neg)
            l_pos = np.sum(f(*inputs_pos).data * R)
            l_neg = np.sum(f(*inputs_neg).data * R)
            analytical = inputs[i].grad[idx]
            numerical = (l_pos - l_neg) / (EPS * 2)
            assert np.allclose(analytical, numerical, rtol=1e-5, atol=1e-6)


array([[0.4046928 , 0.10332152, 0.26530759],
       [0.24099103, 0.10793583, 0.87118127],
       [0.18337666, 0.07391652, 0.53666352]])